# CPU-only Fully Online Two-FFT Inference

This notebook runs every `signal_*.tim` IQ file currently present in CSPB.ML 2022 Batch 28. The runtime needs only **PyTorch, NumPy, and the Python standard library**. It does not import scikit-learn, joblib, pandas, tqdm, or matplotlib.

Pipeline: `raw IQ -> blind BOI filtering/shift -> P96 clipping -> Hann FFT(z^2, z^4) -> frozen NumPy LR8_24 -> QPSK-like CC/CAPNetLite refinement`.

The trained LR parameters are frozen in `helper_function/inference.py`; the `.joblib` file is therefore not required at runtime. CAPNetLite is loaded from its PyTorch checkpoint with `map_location="cpu"`.

In [1]:
from pathlib import Path
import importlib
import sys

import numpy as np
import torch

# Supports starting Jupyter either in this notebook directory or at repo root.
base_candidates = (Path.cwd(), Path.cwd() / "SPL_code" / "2fft_inference")
BASE_DIR = next(
    (path for path in base_candidates if (path / "helper_function").is_dir()),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError(
        "Start Jupyter in the notebook directory or the SPL_Surreal repository root"
    )

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import helper_function.inference as inference_module
inference_module = importlib.reload(inference_module)
CLASS_NAMES = inference_module.CLASS_NAMES
FullyOnlineZPipeline = inference_module.FullyOnlineZPipeline
run_batch_in_memory = inference_module.run_batch_in_memory

BATCH_DIR = BASE_DIR / "dataset" / "Batch_Dir_28"
TRUTH_PATH = BASE_DIR / "dataset" / "CSPB_ML_2022_Signal_Truth_Labels.txt"
CAP_MODEL_PATH = (
    BASE_DIR
    / "models"
    / "capnet_lite_lr8_24_zrawcc_warpscale_k5_c20_20_nb3_2_2018train20.pt"
)

for required_path in (BATCH_DIR, TRUTH_PATH, CAP_MODEL_PATH):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

signal_paths = list(BATCH_DIR.glob("signal_*.tim"))
if not signal_paths:
    raise FileNotFoundError(f"No signal_*.tim files found in {BATCH_DIR}")
print(f"Found {len(signal_paths):,} signals")
print(f"PyTorch {torch.__version__}; NumPy {np.__version__}; device=cpu")

Found 16 signals
PyTorch 2.5.1+cu121; NumPy 1.26.4; device=cpu


## Run complete inference

Every signal currently found in the batch directory is evaluated on CPU, with no SNR-based filtering. A progress line is printed every 100 signals.

In [2]:
pipeline = FullyOnlineZPipeline(
    cap_model_path=CAP_MODEL_PATH,
    device="cpu",
)
results = run_batch_in_memory(
    pipeline=pipeline,
    batch_dir=BATCH_DIR,
    truth_path=TRUTH_PATH,
    expected_count=None,
    progress_every=100,
)
print(f"Completed {len(results):,} signals on {pipeline.device.type}")

CPU inference: 16/16 signals
Completed 16 signals on cpu


## Accuracy and confusion matrix

The confusion matrix is calculated and printed with NumPy. Rows are true labels and columns are predictions.

In [3]:
true_labels = results["true_label"]
predictions = results["prediction"]
overall_accuracy = float(np.mean(true_labels == predictions))

label_to_index = {label: index for index, label in enumerate(CLASS_NAMES)}
matrix = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
for true_label, prediction in zip(true_labels, predictions):
    matrix[label_to_index[true_label], label_to_index[prediction]] += 1

print(f"Overall accuracy: {overall_accuracy:.4%}")
column_width = max(9, max(map(len, CLASS_NAMES)) + 1)
print("true\\pred".rjust(column_width), end="")
for label in CLASS_NAMES:
    print(label.rjust(column_width), end="")
print()
for label, row in zip(CLASS_NAMES, matrix):
    print(label.rjust(column_width), end="")
    for value in row:
        print(str(int(value)).rjust(column_width), end="")
    print()

Overall accuracy: 93.7500%
true\pred     BPSK      MSK    DQPSK     8PSK     QPSK    16QAM    64QAM   256QAM
     BPSK        2        0        0        0        0        0        0        0
      MSK        0        2        0        0        0        0        0        0
    DQPSK        0        0        2        0        0        0        0        0
     8PSK        0        0        0        2        0        0        0        0
     QPSK        0        0        0        0        2        0        0        0
    16QAM        0        0        0        0        0        2        0        0
    64QAM        0        0        0        0        0        0        1        1
   256QAM        0        0        0        0        0        0        0        2
